In [ ]:

import os, sys, glob, shutil, time, json
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_human.npy", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
for p in glob.glob(base+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.hybrid import product_disjoint_pair_masks
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from src.attr_features import FEATURE_NAMES
from src.name_features import NAME_FEATURE_NAMES
from src.string_features import STRING_FEATURE_NAMES
from src.neighbour_features import NEIGHBOUR_FEATURE_NAMES
from src.brand_features import BRAND_FEATURE_NAMES
from src.dim_features import DIM_FEATURE_NAMES
ALL=FEATURE_NAMES+NAME_FEATURE_NAMES+STRING_FEATURE_NAMES+NEIGHBOUR_FEATURE_NAMES+BRAND_FEATURE_NAMES+DIM_FEATURE_NAMES
keep=[i for i,n in enumerate(ALL) if n not in set(NEIGHBOUR_FEATURE_NAMES)]
Xh=np.load(prev+"/features_human.npy")[:,keep]; Xl=np.load(prev+"/features_llm.npy")[:,keep]
E1=np.load(prev+"/features_eval_lex.npy")[:,keep]; E2=np.load(prev+"/features_eval_mixed.npy")[:,keep]
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
ev1=pd.read_parquet(base+"/eval_pairs.parquet"); ev2=pd.read_parquet(base+"/eval_pairs_mixed.parquet")
lp=pd.read_parquet(base+"/llm_pairs_sel.parquet")
y=hm["target"].to_numpy(np.int8); yl=lp["label"].to_numpy(np.int8)
tm,vm=product_disjoint_pair_masks(hm["id1"].to_numpy(),hm["id2"].to_numpy(),0,3)
rel=np.flatnonzero(~vm)
c1=ev1["category"].astype(str).to_numpy(); c2=ev2["category"].astype(str).to_numpy()
y1=ev1["target"].to_numpy(np.int8); y2=ev2["target"].to_numpy(np.int8)
def macro(p,c,yy): return float(np.mean([average_precision_score(yy[c==k],p[c==k])
    for k in np.unique(c) if len(np.unique(yy[c==k]))>1]))
log(f"честное обучение без holdout: {len(Xl)+len(rel):,} пар, признаков {Xh.shape[1]}")
m=HistGradientBoostingClassifier(max_iter=800,learning_rate=0.05,max_leaf_nodes=63,
    random_state=0,early_stopping=False).fit(np.vstack([Xl,Xh[rel]]),np.concatenate([yl,y[rel]]))
p1=m.predict_proba(E1)[:,1]; p2=m.predict_proba(E2)[:,1]
np.save("/kaggle/working/honest_lex.npy",p1); np.save("/kaggle/working/honest_mix.npy",p2)
log(f"без окрестностей, честно: лексич {macro(p1,c1,y1):.6f}  смеш {macro(p2,c2,y2):.6f}")
log("для сравнения со 128 признаками: лексич 0.633960  смеш 0.718292")
log("готово")
